# Hands-on 4: Subscribe Data MQTT dengan Python

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 3.

Tujuan:
- Terhubung ke broker MQTT sebagai subscriber.
- Berlangganan satu topik dan topik wildcard (`kampus/lab1/+/telemetry`).
- Menerima pesan real-time dan mengubah payload JSON menjadi objek Python.
- Melakukan validasi awal data (kelengkapan field dan rentang nilai).

Prasyarat: publisher Hands-on 3 (`scripts/mqtt/publish_sensor.py`) dan broker MQTT lokal.
Notebook ini menjalankan broker lokal berbasis Python (`scripts/mqtt/run_broker.py`) agar alur lengkap dapat dieksekusi tanpa instalasi sistem tambahan.

In [1]:
import os
from pathlib import Path

# Pastikan working directory adalah root repositori,
# karena nbconvert mengeksekusi notebook dari folder notebooks/.
while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /home/sandbox/iot-ml-book


In [2]:
import json
import subprocess
import sys
import time
from pathlib import Path

PYTHON = sys.executable
BROKER = "127.0.0.1"
PORT = 1883
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

print("[INFO] Utilitas siap. Python:", PYTHON)

[INFO] Utilitas siap. Python: /usr/bin/python3


## 1. Menjalankan Broker MQTT Lokal

Buku memakai Mosquitto (`mosquitto -v`). Di sini kita pakai broker lokal berbasis Python
yang setara (`scripts/mqtt/run_broker.py`) pada `127.0.0.1:1883`.

In [3]:
broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb.log"
)
time.sleep(3)
print("[INFO] Broker PID:", broker_proc.pid)
print("[INFO] Broker berjalan:", broker_proc.poll() is None)

[INFO] Broker PID: 10617
[INFO] Broker berjalan: True


## 2. Menjalankan Publisher (Hands-on 3)

Tiga publisher mengirim telemetri ke `kampus/lab1/room01/telemetry`, `room02`, dan `room03`
agar subscriber wildcard dapat didemonstrasikan.

In [4]:
# Publisher buku (scripts/mqtt/publish_sensor.py) dijalankan apa adanya: while True, interval 5 detik.
pub = latar("scripts/mqtt/publish_sensor.py", "publisher_room01_nb.log")
print("[INFO] Publisher room01 PID:", pub[0].pid)
time.sleep(1)

[INFO] Publisher room01 PID: 10618


## 3. Subscriber Sederhana (Satu Topik)

`scripts/mqtt/subscribe_sensor.py` berlangganan `kampus/lab1/room01/telemetry`,
menampilkan payload mentah, dan memparsing field JSON.

In [5]:
# Subscriber buku dijalankan apa adanya (loop_forever); notebook menghentikannya setelah 14 detik.
print(jalankan_buku("scripts/mqtt/subscribe_sensor.py", 14))

Terhubung ke broker MQTT.
Subscribe ke topik: kampus/lab1/room01/telemetry

=== Pesan baru diterima ===
Topik   : kampus/lab1/room01/telemetry
QoS     : 1
Payload : {"device_id": "room01", "temperature_c": 27.34, "humidity_pct": 51.16, "co2_ppm": 721, "power_watt": 139.11, "timestamp": "2026-10-22T13:20:37"}
Payload JSON valid.
Device ID     : room01
Temperature C : 27.34
Humidity %    : 51.16
CO2 ppm       : 721
Power watt    : 139.11
Timestamp     : 2026-10-22T13:20:37

=== Pesan baru diterima ===
Topik   : kampus/lab1/room01/telemetry
QoS     : 1
Payload : {"device_id": "room01", "temperature_c": 28.74, "humidity_pct": 50.86, "co2_ppm": 454, "power_watt": 155.78, "timestamp": "2026-10-22T13:20:42"}
Payload JSON valid.
Device ID     : room01
Temperature C : 28.74
Humidity %    : 50.86
CO2 ppm       : 454
Power watt    : 155.78
Timestamp     : 2026-10-22T13:20:42



## 4. Subscriber Wildcard untuk Banyak Perangkat

`scripts/mqtt/subscribe_all_rooms.py` memakai topik `kampus/lab1/+/telemetry`
sehingga membaca room01, room02, dan room03 sekaligus.

In [6]:
# Subscriber wildcard buku (kampus/lab1/+/telemetry), dihentikan setelah 14 detik.
# Selain room01 dari publisher di atas, room02 dan room03 dikirim sekali agar wildcard terlihat bekerja.
import paho.mqtt.publish as mqttpub
import threading
def _kirim_room_lain():
    time.sleep(4)
    for room in ["room02", "room03"]:
        mqttpub.single(f"kampus/lab1/{room}/telemetry",
                       json.dumps({"device_id": room, "temperature_c": 27.5, "humidity_pct": 55.0,
                                   "co2_ppm": 600, "power_watt": 130.0, "timestamp": "2026-10-22T00:00:00"}),
                       qos=1, hostname=BROKER, port=PORT, client_id=f"uji-{room}")
threading.Thread(target=_kirim_room_lain).start()
print(jalankan_buku("scripts/mqtt/subscribe_all_rooms.py", 14))

Terhubung ke broker.
Subscribe wildcard ke: kampus/lab1/+/telemetry
[kampus/lab1/room02/telemetry] {'device_id': 'room02', 'temperature_c': 27.5, 'humidity_pct': 55.0, 'co2_ppm': 600, 'power_watt': 130.0, 'timestamp': '2026-10-22T00:00:00'}
[kampus/lab1/room03/telemetry] {'device_id': 'room03', 'temperature_c': 27.5, 'humidity_pct': 55.0, 'co2_ppm': 600, 'power_watt': 130.0, 'timestamp': '2026-10-22T00:00:00'}
[kampus/lab1/room01/telemetry] {'device_id': 'room01', 'temperature_c': 26.63, 'humidity_pct': 51.03, 'co2_ppm': 568, 'power_watt': 140.71, 'timestamp': '2026-10-22T13:20:52'}
[kampus/lab1/room01/telemetry] {'device_id': 'room01', 'temperature_c': 26.54, 'humidity_pct': 62.8, 'co2_ppm': 590, 'power_watt': 159.98, 'timestamp': '2026-10-22T13:20:57'}



## 5. Subscriber dengan Validasi Payload

`scripts/mqtt/subscribe_validated.py` memeriksa kelengkapan field dan rentang nilai:
`suhu 0-60 C`, `kelembaban 0-100 %`, `CO2 300-5000 ppm`.

Untuk mendemonstrasikan deteksi data tidak valid, subscriber dijalankan di latar
belakang, lalu dua payload rusak (suhu di luar rentang dan payload non-JSON) dikirim
ke topik yang sama di tengah arus data normal.

In [7]:
# Kirim payload tidak valid SAAT subscriber validasi berjalan.
import paho.mqtt.publish as mqttpub

def kirim_payload_rusak():
    payload_rusak = {
        "device_id": "room03",
        "temperature_c": 95.0,  # di luar rentang valid 0-60
        "humidity_pct": 50.0,
        "co2_ppm": 700,
        "power_watt": 120.0,
        "timestamp": "2026-10-22T00:00:00",
    }
    mqttpub.single(
        "kampus/lab1/room03/telemetry",
        json.dumps(payload_rusak),
        qos=1, hostname=BROKER, port=PORT,
        client_id="publisher-anomali",
    )
    mqttpub.single(
        "kampus/lab1/room03/telemetry",
        "bukan-json",
        qos=1, hostname=BROKER, port=PORT,
        client_id="publisher-anomali",
    )
    print("[INFO] 2 payload uji (tidak valid) dikirim.")

In [8]:
# Jalankan subscriber validasi di latar belakang,
# lalu kirim payload rusak di tengah arus data normal.
val_proc, val_f = start_process(
    [
        PYTHON, "scripts/mqtt/subscribe_validated.py",
        "--timeout", "10",
    ],
    LOG_DIR / "subscriber_validated_nb_run.log",
)
time.sleep(2)  # beri waktu subscriber tersambung dan subscribe
kirim_payload_rusak()
val_proc.wait(timeout=60)
val_f.close()
print((LOG_DIR / "subscriber_validated_nb_run.log").read_text(encoding="utf-8"))

[INFO] 2 payload uji (tidak valid) dikirim.


[INFO] Terhubung ke broker.
[INFO] Subscribe dengan validasi ke: kampus/lab1/+/telemetry
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 29.92, 'humidity_pct': 56.45, 'co2_ppm': 468, 'power_watt': 151.64, 'timestamp': '2026-10-22T13:21:02'}
[PERINGATAN] Data tidak valid: {'device_id': 'room03', 'temperature_c': 95.0, 'humidity_pct': 50.0, 'co2_ppm': 700, 'power_watt': 120.0, 'timestamp': '2026-10-22T00:00:00'}
[ERROR] Error parsing payload: Expecting value: line 1 column 1 (char 0)
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 25.49, 'humidity_pct': 56.21, 'co2_ppm': 515, 'power_watt': 102.57, 'timestamp': '2026-10-22T13:21:07'}
[VALID] Data valid: {'device_id': 'room01', 'temperature_c': 28.86, 'humidity_pct': 53.38, 'co2_ppm': 868, 'power_watt': 107.95, 'timestamp': '2026-10-22T13:21:12'}
[INFO] Ringkasan: 5 pesan, 3 valid, 2 tidak valid.



## 6. Verifikasi Log dan Pembersihan

Semua subscriber menulis ringkasan ke `outputs/logs/`. Di akhir sesi,
publisher dan broker dihentikan.

In [9]:
henti(pub)
broker_proc.terminate()
broker_f.close()
time.sleep(1)
print("[INFO] Publisher dan broker dihentikan.")

for log_name in ["subscriber_room01.log", "subscriber_all_rooms.log", "subscriber_validated.log"]:
    log_path = LOG_DIR / log_name
    print(f"\n===== {log_name} (5 baris terakhir) =====")
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8").strip().splitlines()
        for line in lines[-5:]:
            print(line)
    else:
        print("[PERINGATAN] Log belum ada.")

[INFO] Publisher dan broker dihentikan.

===== subscriber_room01.log (5 baris terakhir) =====
2026-10-22T07:44:53.722145 - Pesan valid dari kampus/lab1/room01/telemetry: {"device_id": "room01", "temperature_c": 24.09, "humidity_pct": 51.25, "co2_ppm": 520, "power_watt": 146.73, "timestamp": "2026-10-22T07:44:53"}
2026-10-22T10:52:20.974656 - Subscribe ke topik: capstone/iotnode01/alert
2026-10-22T10:52:23.148474 - Pesan valid dari capstone/iotnode01/alert: {"timestamp": "2026-10-22T10:52:23", "device_id": "iotnode01", "alert_reason": "isolation_forest", "temperature_c": 33.76, "humidity_pct": 41.63, "power_watt": 234.52, "current_amp": 5.24}
2026-10-22T12:04:16.715169 - Subscribe ke topik: iotlab/room01/alert
2026-10-22T12:04:20.709700 - Pesan valid dari iotlab/room01/alert: {"timestamp": "2026-10-22T12:04:20", "device_id": "room01", "pred_status": "fault", "alert_reason": "Model memprediksi FAULT", "temperature_c": 25.46, "humidity_pct": 52.42, "co2_ppm": 1230, "power_watt": 259.06}



## Penutup

Subscriber Python kini menjadi "mata digital" yang membaca arus telemetri MQTT secara
real-time, memvalidasi payload, dan menyiapkan data untuk tahap penyimpanan
(Hands-on 5: CSV / SQLite / InfluxDB).

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 4 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 7. Langkah kerja yang harus dilakukan pembaca

Tahap

Kegiatan

Jalankan broker.

Jalankan publisher.

Jalankan subscriber sederhana.

Ubah subscriber agar memakai wildcard.

Tambahkan validasi payload.

Siapkan subscriber untuk tahap penyimpanan data.